In [1]:
library(tidyverse)

library(grid)
library(gridExtra)
library(patchwork)

source("../evaluation_utils/plots_eda.R")
source("../evaluation_utils/evaluation_funcs.R")

Warning message:
“package ‘dplyr’ was built under R version 4.5.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine


Loading required package: viridisLite

Loading required package: limma

Loading required package: BiocParallel


Attaching package: ‘variancePartition’


The following objects are masked from ‘package:limma’:

    eBayes, topTable




In [2]:
cbPalette <- c("#CC79A7", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00")

# Load data

central_corrected === fedRBE corrected data

In [ ]:
data_path <- "../evaluation_data/ccRCC_studies/"

central_uncorrected <- read.csv(
    paste0(data_path, "before/central_intensities_log_UNION.tsv"),
    sep = "\t", header = TRUE, row.names = 1, check.names = FALSE
)

# Load combined metadata from ccRCC_metadata.csv.
# Columns: Sample (sample ID), Condition (Tumor/Normal), Dataset (study/batch).
metadata_raw <- read.csv(
    paste0(data_path, "data/ccRCC_metadata.csv"),
    header = TRUE, row.names = 1
)
metadata <- metadata_raw %>%
    rename(file = Sample, condition = Condition, lab = Dataset) %>%
    column_to_rownames("file")
metadata$file <- rownames(metadata)

central_corrected <- read.csv(
    paste0(data_path, "after/intensities_log_Rcorrected_UNION.tsv"),
    sep = "\t", header = TRUE, row.names = 1, check.names = FALSE
)
fed_corrected <- read.csv(
    paste0(data_path, "after/FedApp_corrected_data.tsv"),
    sep = "\t", header = TRUE, row.names = 1, check.names = FALSE
)

# Align matrices to metadata sample order
central_corrected <- central_corrected[, rownames(metadata)]
central_uncorrected <- central_uncorrected[rownames(central_corrected), rownames(metadata)]
fed_corrected <- fed_corrected[rownames(central_corrected), rownames(metadata)]

cat("Loaded data\n")
cat("Central uncorrected:", nrow(central_uncorrected), "rows\n")
cat("Central corrected:", nrow(central_corrected), "rows\n")
cat("Fed corrected:", nrow(fed_corrected), "rows\n")
cat("Sample metadata:", nrow(metadata), "rows\n\n")

# Checks

## Diagnostic plots

In [ ]:
pca_plot_uncorrected <- pca_plot(central_uncorrected, metadata,
    title = "Before correction",
    quantitative_col_name = "file", col_col = "condition", shape_col = "lab",
    show_legend = FALSE, cbPalette = c("#44abe7", "#c55702"))
pca_plot_fed <- pca_plot(fed_corrected, metadata,
    title = "After FedRBE correction",
    quantitative_col_name = "file", col_col = "condition", shape_col = "lab",
    show_legend = TRUE, cbPalette = c("#44abe7", "#c55702"))

layout <- (pca_plot_uncorrected + pca_plot_fed)
layout <- layout + plot_annotation("PCA plot, ccRCC proteomics data")
options(repr.plot.width = 6, repr.plot.height = 3)
layout

In [ ]:
y_min <- min(central_uncorrected, fed_corrected, na.rm = TRUE)
y_max <- max(central_uncorrected, fed_corrected, na.rm = TRUE)
y_limits <- c(y_min, y_max)

boxplots_uncorrected <- boxplot_plot_groupped(central_uncorrected, metadata,
    title = "Before correction",
    quantitativeColumnName = "file",
    color_col = "lab", remove_xnames = TRUE, show_legend = FALSE,
    y_limits = y_limits, cbPalette = cbPalette)

boxplots_fed <- boxplot_plot_groupped(fed_corrected, metadata,
    title = "After fedRBE correction",
    quantitativeColumnName = "file",
    color_col = "lab", remove_xnames = TRUE,
    y_limits = y_limits, cbPalette = cbPalette)

layout <- (boxplots_uncorrected + boxplots_fed)
layout <- layout + plot_annotation("Violin plots, ccRCC proteomics data")
options(repr.plot.width = 6, repr.plot.height = 3)
layout

## Linear model per variable

Following batch effect correction, the percentage of variance explained by the condition should be greater than the batch.

In [ ]:
library(variancePartition)

In [ ]:
form <- ~ condition + lab

In [ ]:
lmpv_plot_fed <- lmpv_plot(fed_corrected, metadata,
    title = "After FedRBE correction", form = form)
lmpv_plot_uncorrected <- lmpv_plot(central_uncorrected, metadata,
    title = "Before correction", show_legend = FALSE, form = form)

layout <- (lmpv_plot_uncorrected + lmpv_plot_fed)
layout <- layout + plot_annotation("Variance partitioning, ccRCC proteomics data")
options(repr.plot.width = 6, repr.plot.height = 3)
layout

# Session info

In [ ]:
sessionInfo()